# **Install Libraries**

In [7]:
!pip uninstall -y datasets
!pip install datasets==2.19.1

Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


# **Task 1 : Dataset Selection**



In [1]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


# **Task 2: Data Preprocessing**

# Step 1 — Load Tokenize

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Step 2 — Label Alignment Function

In [3]:
def tokenize_and_align_labels(examples, label_name):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples[label_name]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Step 3 — Apply for POS and Chunking

In [4]:
pos_tokenized = dataset.map(
    lambda x: tokenize_and_align_labels(x, "pos_tags"),
    batched=True
)

chunk_tokenized = dataset.map(
    lambda x: tokenize_and_align_labels(x, "chunk_tags"),
    batched=True
)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

# Step 4 — Check Output

In [5]:
print(pos_tokenized["train"][0])
print(chunk_tokenized["train"][0])

{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0], 'input_ids': [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 22, 42, 16, 21, 35, 37, 16, 21, 21, 7, -100]}
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0], 'input_ids': [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 11, 21, 11, 12, 21, 22, 11, 12, 12, 0, -

# **Task 3: Model Setup**

# Step 1 — Get number of labels

In [6]:
num_pos_labels = len(dataset["train"].features["pos_tags"].feature.names)
num_chunk_labels = len(dataset["train"].features["chunk_tags"].feature.names)

print(num_pos_labels, num_chunk_labels)

47 23


# Step 2 — Load Model for POS Tagging

In [7]:
from transformers import AutoModelForTokenClassification

pos_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=num_pos_labels
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

# Step 3 — Load Model for Chunking

In [8]:
chunk_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=num_chunk_labels
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

# Step 4 — Verify Models

In [9]:
print(pos_model)
print(chunk_model)

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

# **Task 4: Training**

# Step 1 — Import

In [10]:
from transformers import TrainingArguments, Trainer
import numpy as np

# Step 2 — Define Metrics

In [11]:
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = [
        [p for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [l for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {"accuracy": metric.compute(predictions=true_predictions, references=true_labels)}

# Step 3 — Training Arguments for POS

In [14]:
from transformers import TrainingArguments

pos_args = TrainingArguments(
    output_dir="./pos_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
)

# Step - 4 POS Trainer

**Step 4.1 — Small Subset**

In [19]:
small_train = pos_tokenized["train"].shuffle(seed=42).select(range(2000))
small_val = pos_tokenized["validation"].shuffle(seed=42).select(range(500))

**Step 4.2 — Data Collator**

In [20]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

**Step 4.3 — Training Arguments**

In [21]:
from transformers import TrainingArguments

pos_args = TrainingArguments(
    output_dir="./pos_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50
)

**Step 4.4 — Trainer Setup**

In [22]:
from transformers import Trainer

pos_trainer = Trainer(
    model=pos_model,
    args=pos_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
)

**Step 4.5 — Start Training**

In [23]:
pos_trainer.train()

Epoch,Training Loss,Validation Loss
1,0.653066,0.600674


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=250, training_loss=1.153208999633789, metrics={'train_runtime': 961.1415, 'train_samples_per_second': 2.081, 'train_steps_per_second': 0.26, 'total_flos': 44634573057792.0, 'train_loss': 1.153208999633789, 'epoch': 1.0})

**Step 4.6 — Save Model**

In [24]:
pos_trainer.save_model("./pos_model")
tokenizer.save_pretrained("./pos_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./pos_model/tokenizer_config.json', './pos_model/tokenizer.json')

# **Task 5: Evaluation**

# Step 5.1 — Install seqeval

In [25]:
!pip install seqeval

# Step 2 — Import Metric

In [26]:
from datasets import load_metric

metric = load_metric("seqeval")

/tmp/ipykernel_7786/3097260500.py:3: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("seqeval")
/usr/local/lib/python3.12/dist-packages/datasets/load.py:759: FutureWarning: The repository for seqeval contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.1/metrics/seqeval/seqeval.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


# Step 3 — Predictions

In [27]:
predictions, labels, _ = pos_trainer.predict(small_val)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [30]:
pos_id2label = pos_model.config.id2label
pos_label2id = pos_model.config.label2id

# Step 4 — Convert IDs to Labels

In [31]:
import numpy as np

preds = np.argmax(predictions, axis=2)

true_labels = [
    [pos_id2label[l] for l in label if l != -100]
    for label in labels
]

true_preds = [
    [pos_id2label[p] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(preds, labels)
]

# step - 6 Metrics

In [32]:
results = metric.compute(predictions=true_preds, references=true_labels)

print("Precision:", results["overall_precision"])
print("Recall:", results["overall_recall"])
print("F1 Score:", results["overall_f1"])

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LABEL_16 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LABEL_24 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LABEL_15 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LABEL_22 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LABEL_41 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequ

Precision: 0.8442565186751233
Recall: 0.8564483843294253
F1 Score: 0.850308751508269


# **Task 6: Inference**

# Step 1 — Load Model & Tokenizer

In [33]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "./pos_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# Step 2 — Test Sentence

In [34]:
sentence = "John works at Google in California"

tokens = tokenizer(sentence, return_tensors="pt")

# Step 3 - Prediction

In [35]:
import torch

with torch.no_grad():
    outputs = model(**tokens)

predictions = torch.argmax(outputs.logits, dim=2)

# Step 4 — Convert IDs to POS Tags

In [36]:
id2label = model.config.id2label

predicted_labels = [
    id2label[p.item()] for p in predictions[0]
]

token_list = tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])

for token, label in zip(token_list, predicted_labels):
    print(token, "→", label)

[CLS] → LABEL_16
John → LABEL_22
works → LABEL_42
at → LABEL_15
Google → LABEL_22
in → LABEL_15
California → LABEL_22
[SEP] → LABEL_7


# **Task 7: Comparison**


## ✅ Task 7 — POS Tagging vs Chunking

### **POS Tagging (Part-of-Speech Tagging)**

* Prathi word ki grammatical role assign chestundi
* Word level tagging
* Grammar understanding kosam use avuthundi

**Example:**

| Word  | POS Tag     |
| ----- | ----------- |
| John  | Proper Noun |
| works | Verb        |
| at    | Preposition |

👉 Focus: **individual words**

---

### **Chunking (Phrase Detection)**

* Words ni group chesi phrases create chestundi
* Phrase level understanding
* Sentence structure ni identify chestundi

**Example:**

| Words           | Chunk                     |
| --------------- | ------------------------- |
| John            | Noun Phrase (NP)          |
| works at Google | Verb Phrase (VP)          |
| in California   | Prepositional Phrase (PP) |

👉 Focus: **group of words (phrases)**

---

### 🔍 Key Differences

| Feature    | POS Tagging     | Chunking           |
| ---------- | --------------- | ------------------ |
| Level      | Word level      | Phrase level       |
| Output     | Noun, Verb, Adj | NP, VP, PP         |
| Complexity | Easy            | Medium             |
| Use        | Grammar         | Sentence structure |
| Example    | John → Noun     | John → NP          |

---

### 🧠 Observation

POS tagging helps model understand **what a word is**.
Chunking helps model understand **how words form meaningful phrases**.


# **Task 8: Report / Blog**


## ✅ Task 8 — Blog / Report Content (ready to paste)

### Challenges Faced

* Tokenization & label alignment for subwords
* Handling `-100` for special tokens
* Training BERT on CPU was slow
* Dataset loading issues due to version conflicts

### Observations & Insights

* Token classification requires careful preprocessing
* Label alignment is the most critical step
* Even small subset training gives good F1 score
## * POS tagging is simpler compared to **chunking** **bold text**